<div style="display: flex; align-items: center;">

  <!-- Logos -->
  <div style="white-space: nowrap;">
    <img 
      src="https://www.upc.edu/premi-secundaria/++theme++genweb6.theme/img/logo.png" 
      width="300"
      style="vertical-align: middle;"
    >
    <img 
      src="https://www.hipotecalowcost.com/wp-content/uploads/2019/08/Logo-CaixaBank.png" 
      width="200"
      style="vertical-align: middle;"
    >
  </div>

  <!-- Texto -->
  <div style="margin-left: auto; margin-right: 10px; text-align: right;">
      <p style="margin: 0;"><b>CaixaBank · Advanced Analytics Program</b></p>
      <p style="margin: 0;"><b>Model Risk & Data Science Training</b></p>
      <p style="margin: 0;">Automatización y robotización de procesos</p>
  </div>

</div>

# Bloque 11 · Automatización de PowerPoint

Ejecuta desde la carpeta del bloque tras instalar requirements.txt. Lee la teoría, ejecuta cada ejemplo y compara la salida con la interpretación.

In [ ]:
from pathlib import Path
from tempfile import TemporaryDirectory
from pptx import Presentation
from pptx.util import Inches, Pt
from presentacion import (nueva_presentacion, pagina, texto, grafico_estados,
                          cola_revision, construir, guardar)
from informe import cargar, resumir

base = Path(__file__).resolve().parent if "__file__" in globals() else Path.cwd()
ejemplos = base / "ejemplos"
documentos = cargar(ejemplos / "indice_pdf.json")
kpi = resumir(documentos)


## Ejemplo 1 · Inspeccionar el lienzo y los diseños

El patrón define elementos heredados y un diseño proporciona una distribución. Las coordenadas se miden desde la esquina superior izquierda. Inches convierte pulgadas en unidades internas.

In [ ]:
prs = nueva_presentacion()
print("Lienzo:", prs.slide_width.inches, prs.slide_height.inches)
for i, layout in enumerate(prs.slide_layouts):
    print(i, layout.name)
slide = prs.slides.add_slide(prs.slide_layouts[0])
print([(p.placeholder_format.idx, p.name) for p in slide.placeholders])

**Interpretación.** El lienzo es cercano a 16:9. Los índices se refieren a la plantilla que acabas de abrir. La lista final muestra los marcadores del primer diseño.

## Ejemplo 2 · Construir y estilizar texto

Una caja contiene un marco de texto, párrafos y tramos. El tramo permite cambiar el estilo de una parte del texto.

In [ ]:
prs = nueva_presentacion()
slide = pagina(prs, "Resumen", "PRÁCTICA")
caja = texto(slide, "Cobertura de extracción", 1, 2, 10, 1, 26)
tramo = caja.text_frame.paragraphs[0].runs[0]
tramo.font.bold = True
print(caja.text, tramo.font.size.pt, tramo.font.bold)

**Interpretación.** La salida confirma el contenido y que el primer tramo tiene 26 puntos y negrita. Esto verifica el estilo almacenado. La lectura en pantalla comprueba si el texto cabe.

## Ejemplo 3 · Insertar evidencia visual

Dar solo el ancho a add_picture conserva la proporción original. El objeto queda situado mediante x e y.

In [ ]:
prs = nueva_presentacion()
slide = pagina(prs, "Evidencia", "PDF DE AULA")
imagen = slide.shapes.add_picture(str(ejemplos / "pagina_pdf.png"),
                                 Inches(1), Inches(2), width=Inches(6))
print("Ancho y alto:", imagen.width.inches, imagen.height.inches)
assert imagen.top + imagen.height <= prs.slide_height

**Interpretación.** El alto se deriva de la imagen. La comprobación confirma que su borde inferior queda dentro del lienzo. La vista final permite revisar la nitidez del texto.

## Ejemplo 4 · Representar la procedencia con una tabla

El gráfico ayuda a comparar cantidades. La tabla permite localizar el documento y la página sobre los que hay que actuar.

In [ ]:
pendientes = cola_revision(documentos)
prs = nueva_presentacion()
slide = pagina(prs, "Páginas por revisar", "PROCEDENCIA")
tabla = slide.shapes.add_table(len(pendientes) + 1, 2, Inches(1), Inches(2),
                              Inches(10), Inches(2)).table
for fila, valores in enumerate([("Archivo", "Página"), *pendientes]):
    for columna, valor in enumerate(valores):
        tabla.cell(fila, columna).text = str(valor)
print([(tabla.cell(i, 0).text, tabla.cell(i, 1).text)
       for i in range(1, len(tabla.rows))])

**Interpretación.** Las dos filas corresponden a la página 1 del escaneado y la página 2 del mixto. La cabecera ocupa la fila cero de la tabla.

## Ejemplo 5 · Construir un gráfico editable

Cada categoría debe corresponder a un valor de la serie. El gráfico incorpora los datos para poder editarlos en PowerPoint.

In [ ]:
prs = nueva_presentacion()
slide = pagina(prs, "Cobertura", "ESTADOS")
grafico = grafico_estados(slide, kpi["con_texto"], kpi["pendientes"])
print(list(grafico.series[0].values))
assert sum(grafico.series[0].values) == kpi["paginas"]

**Interpretación.** La serie contiene dos páginas con texto y dos pendientes. La suma coincide con el universo observado, cuatro páginas.

## Ejemplo 6 · Paginar el detalle y tratar el vacío

Cada diapositiva de revisión admite ocho casos. Con una cola vacía, la diapositiva muestra que el lote no tiene páginas pendientes.

In [ ]:
prs = construir(documentos, "Aula")
vacia = construir([], "Sin registros")
print("Diapositivas del lote:", len(prs.slides))
print([s.text for s in vacia.slides[2].shapes if s.has_text_frame])

**Interpretación.** El lote de aula produce tres diapositivas. La cobertura sin registros aparece como No aplica. Una cola vacía puede coexistir con excepciones de archivo, por eso la portada declara cifrados y errores.

## Ejemplo 7 · Guardar, reabrir y reconciliar

Primero comprobamos la estructura del archivo generado. Después comparamos el gráfico y la tabla con el cálculo original.

In [ ]:
with TemporaryDirectory() as temporal:
    ruta = Path(temporal) / "observatorio.pptx"
    guardar(construir(documentos, "Aula"), ruta)
    revisada = Presentation(ruta)
    serie = next(s.chart.series[0] for s in revisada.slides[1].shapes if s.has_chart)
    assert list(serie.values) == [kpi["con_texto"], kpi["pendientes"]]
    print("PPTX reabierto. Diapositivas:", len(revisada.slides))

**Interpretación.** La reapertura confirma que las cifras siguen en el gráfico. Para una entrega operativa, abre el resultado en la aplicación de destino y verifica composición y texto.

## Diez ejercicios

Completa cada celda y justifica el resultado. El solucionario incluye las comprobaciones.

### Ejercicio 01 · Comprobar el formato del lienzo

Crea una presentación y comprueba que su relación de aspecto se aproxima a 16:9.

In [ ]:
# Escribe aquí tu solución.

### Ejercicio 02 · Leer una plantilla

Abre el PPTX de ejemplo. Muestra cuántas diapositivas contiene y lista los nombres de sus diseños.

In [ ]:
# Escribe aquí tu solución.

### Ejercicio 03 · Editar un marcador

Crea una diapositiva con el primer diseño de la plantilla predeterminada. Escribe y recupera el título.

In [ ]:
# Escribe aquí tu solución.

### Ejercicio 04 · Estilizar una caja de texto

Escribe un subtítulo con 24 puntos y negrita. Comprueba tamaño y estilo.

In [ ]:
# Escribe aquí tu solución.

### Ejercicio 05 · Comprobar la geometría de una imagen

Inserta la captura PDF con seis pulgadas de ancho. Comprueba los bordes derecho e inferior.

In [ ]:
# Escribe aquí tu solución.

### Ejercicio 06 · Auditar la tabla de procedencia

Localiza la tabla de la tercera diapositiva y compara sus filas con cola_revision.

In [ ]:
# Escribe aquí tu solución.

### Ejercicio 07 · Verificar los datos del gráfico

Recupera la serie y verifica las dos cantidades y su suma.

In [ ]:
# Escribe aquí tu solución.

### Ejercicio 08 · Ensayar una cola larga

Genera nueve páginas pendientes sintéticas y comprueba que se reparten en dos diapositivas de detalle.

In [ ]:
# Escribe aquí tu solución.

### Ejercicio 09 · Interpretar el denominador vacío

Genera el informe de un lote vacío. Comprueba cómo se muestra la cobertura y explica por qué.

In [ ]:
# Escribe aquí tu solución.

### Ejercicio 10 · Reabrir la entrega final

Guarda en temporal, reabre y verifica que todas las formas están dentro del lienzo.

In [ ]:
# Escribe aquí tu solución.